# Cell 1 — Setup

In [3]:
from pathlib import Path
import sys
import json
import pickle

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# If the notebook is inside a subfolder, add project root to path.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "generator").exists():
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

from generator.features_config import (
    PREDICTION_CUTOFF,
    FEATURES_PATH,
    FEATURES_GRAPH_PATH,
    GROUND_TRUTH_PATH,
    BASELINE_METRICS_PATH,
    MODEL_DIR,
    PATHS,
)
from generator.config import SEED

print("Project root:", PROJECT_ROOT)

Project root: d:\CODIN PLAYGROUND\ML-AI\RingWatch


# Cell 2 — Constants and output paths


In [4]:
T = pd.Timestamp(PREDICTION_CUTOFF)

COST_FALSE_POSITIVE = 2000.0   # ₹
COST_FALSE_NEGATIVE = 15000.0  # ₹

RANDOM_STATE = 42

RING_TRAIN = ["R001", "R002"]
RING_VAL = ["R003"]
RING_TEST = ["R004"]

MODEL_PATH_A = MODEL_DIR / "model_lgbm_A.pkl"
MODEL_PATH_B = MODEL_DIR / "model_lgbm_B.pkl"
PREDICTIONS_TEST_PATH = MODEL_DIR / "model_predictions_test.csv"
MODEL_METRICS_PATH = MODEL_DIR / "model_metrics.json"
FEATURE_IMPORTANCE_PATH = MODEL_DIR / "model_feature_importance.csv"
MODEL_LEAKAGE_REPORT_PATH = MODEL_DIR / "model_leakage_report.txt"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Cutoff:", T)
print("Ring train:", RING_TRAIN)
print("Ring validation:", RING_VAL)
print("Ring test:", RING_TEST)

Cutoff: 2026-02-20 00:00:00
Ring train: ['R001', 'R002']
Ring validation: ['R003']
Ring test: ['R004']


# Cell 3 — Load raw feature matrices and ground truth


In [5]:
features_day4 = pd.read_csv(FEATURES_PATH)
features_graph = pd.read_csv(FEATURES_GRAPH_PATH)
ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

print("Day 4 features shape:", features_day4.shape)
print("Day 5 graph features shape:", features_graph.shape)
print("Ground truth shape:", ground_truth.shape)

# Schema checks
assert len(features_day4) == 1000
assert len(features_graph) == 1000
assert len(ground_truth) == 1000

assert features_day4["account_id"].is_unique
assert features_graph["account_id"].is_unique
assert ground_truth["account_id"].is_unique

Day 4 features shape: (1000, 41)
Day 5 graph features shape: (1000, 54)
Ground truth shape: (1000, 6)


In [6]:
# Cell 4 — Merge ground truth


In [7]:
data_day4 = features_day4.merge(
    ground_truth,
    on="account_id",
    how="left",
    validate="one_to_one",
)

data_graph = features_graph.merge(
    ground_truth,
    on="account_id",
    how="left",
    validate="one_to_one",
)

assert data_day4["true_ring_member"].notna().all()
assert data_graph["true_ring_member"].notna().all()

print("Merged Day 4 shape:", data_day4.shape)
print("Merged graph shape:", data_graph.shape)

Merged Day 4 shape: (1000, 46)
Merged graph shape: (1000, 59)


# Cell 5 — Ring-aware split

In [8]:
ring_members = ground_truth[
    ground_truth["true_ring_member"] == True
].copy()

non_ring_members = ground_truth[
    ground_truth["true_ring_member"] == False
].copy()

train_ring_ids = ring_members[
    ring_members["abuse_ring_id"].isin(RING_TRAIN)
]["account_id"].tolist()

val_ring_ids = ring_members[
    ring_members["abuse_ring_id"].isin(RING_VAL)
]["account_id"].tolist()

test_ring_ids = ring_members[
    ring_members["abuse_ring_id"].isin(RING_TEST)
]["account_id"].tolist()

# Non-ring accounts: 50% train, 20% val, 30% test
non_ring_ids = non_ring_members["account_id"].tolist()

normal_train, normal_remain = train_test_split(
    non_ring_ids,
    test_size=0.5,
    random_state=RANDOM_STATE,
    shuffle=True,
)

normal_val, normal_test = train_test_split(
    normal_remain,
    test_size=0.6,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train_ids = pd.Index(train_ring_ids + normal_train)
val_ids = pd.Index(val_ring_ids + normal_val)
test_ids = pd.Index(test_ring_ids + normal_test)

print("Train accounts:", len(train_ids))
print("Validation accounts:", len(val_ids))
print("Test accounts:", len(test_ids))

Train accounts: 506
Validation accounts: 197
Test accounts: 297


# Cell 6 — Split validation


In [9]:
assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)

assert set(train_ring_ids).issubset(set(train_ids))
assert set(val_ring_ids).issubset(set(val_ids))
assert set(test_ring_ids).issubset(set(test_ids))

# R004 must be only in test, etc.
assert set(test_ring_ids).issubset(set(test_ids))
assert not set(test_ring_ids).intersection(set(train_ids))
assert not set(test_ring_ids).intersection(set(val_ids))

split_data = {
    "train": {
        "day4": data_day4[data_day4["account_id"].isin(train_ids)].copy(),
        "graph": data_graph[data_graph["account_id"].isin(train_ids)].copy(),
    },
    "val": {
        "day4": data_day4[data_day4["account_id"].isin(val_ids)].copy(),
        "graph": data_graph[data_graph["account_id"].isin(val_ids)].copy(),
    },
    "test": {
        "day4": data_day4[data_day4["account_id"].isin(test_ids)].copy(),
        "graph": data_graph[data_graph["account_id"].isin(test_ids)].copy(),
    },
}

for split_name, split_dict in split_data.items():
    print(f"\n{split_name}:")
    for feat_name, df in split_dict.items():
        print(
            f"  {feat_name}: rows={len(df)}, "
            f"pos={df['true_ring_member'].sum():.0f}"
        )


train:
  day4: rows=506, pos=28
  graph: rows=506, pos=28

val:
  day4: rows=197, pos=6
  graph: rows=197, pos=6

test:
  day4: rows=297, pos=10
  graph: rows=297, pos=10


# Cell 7 — Feature preparation: Model A

In [10]:
forbidden_columns = [
    "account_id",
    "true_ring_member",
    "abuse_ring_id",
    "ring_type",
    "ring_start_time",
    "ring_end_time",
    "population_type",
]

def get_allowed_feature_cols(df):
    return [
        col
        for col in df.columns
        if col not in forbidden_columns
    ]

train_A_raw = split_data["train"]["day4"]
cols_A_all = get_allowed_feature_cols(train_A_raw)

# Drop constant features based only on training data.
constant_A = [
    col
    for col in cols_A_all
    if train_A_raw[col].nunique(dropna=False) <= 1
]

feature_cols_A = [
    col
    for col in cols_A_all
    if col not in constant_A
]

print("Model A feature count:", len(feature_cols_A))
print("Constant columns removed:", constant_A)

Model A feature count: 38
Constant columns removed: ['refunds_last_24h', 'refund_burst_score']


# Cell 8 — Feature preparation: Model B

In [11]:
train_B_raw = split_data["train"]["graph"]
cols_B_all = get_allowed_feature_cols(train_B_raw)

# Remove arbitrary Louvain label.
cols_B_all = [
    col for col in cols_B_all
    if col != "community_id"
]

constant_B = [
    col
    for col in cols_B_all
    if train_B_raw[col].nunique(dropna=False) <= 1
]

feature_cols_B = [
    col
    for col in cols_B_all
    if col not in constant_B
]

print("Model B feature count:", len(feature_cols_B))
print("Constant columns removed:", constant_B)
print("Remaining B features:", feature_cols_B[:10], "...")

Model B feature count: 50
Constant columns removed: ['refunds_last_24h', 'refund_burst_score']
Remaining B features: ['total_orders', 'total_amount', 'avg_order_value', 'distinct_devices', 'distinct_addresses', 'distinct_phones', 'distinct_payment_instruments', 'total_delivered_orders', 'total_failed_orders', 'total_pending_orders'] ...


# Cell 9 — Build model inputs

In [12]:
inputs = {}

for split_name, split_dict in split_data.items():
    for feat_name, df in split_dict.items():
        if feat_name == "day4":
            cols = feature_cols_A
        else:
            cols = feature_cols_B

        X = df[cols].copy()
        y = df["true_ring_member"].astype(int).copy()

        # Validation
        assert X.select_dtypes(exclude="number").shape[1] == 0
        assert np.isfinite(X.to_numpy()).all()
        assert not X.isna().any().any()

        inputs[f"{split_name}_{feat_name}"] = (X, y)

print("Inputs prepared:")
for key, (X, y) in inputs.items():
    print(f"  {key}: X={X.shape}, positive={y.sum()}")

Inputs prepared:
  train_day4: X=(506, 38), positive=28
  train_graph: X=(506, 50), positive=28
  val_day4: X=(197, 38), positive=6
  val_graph: X=(197, 50), positive=6
  test_day4: X=(297, 38), positive=10
  test_graph: X=(297, 50), positive=10


# Cell 10 — Class imbalance

In [13]:
X_train_A, y_train_A = inputs["train_day4"]
X_train_B, y_train_B = inputs["train_graph"]

# Both use same account split, so labels are identical.
assert (y_train_A.values == y_train_B.values).all()

negatives = (y_train_A == 0).sum()
positives = (y_train_A == 1).sum()
scale_pos_weight = negatives / positives

print("Train negatives:", negatives)
print("Train positives:", positives)
print("scale_pos_weight:", round(scale_pos_weight, 2))

Train negatives: 478
Train positives: 28
scale_pos_weight: 17.07


# Cell 11 — Train Model A

In [14]:
model_A = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    scale_pos_weight=scale_pos_weight,
    verbose=-1,
)

model_A.fit(X_train_A, y_train_A)

print("Model A trained.")

Model A trained.


# Cell 12 — Train Model B

In [15]:
model_B = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    scale_pos_weight=scale_pos_weight,
    verbose=-1,
)

model_B.fit(X_train_B, y_train_B)

print("Model B trained.")

Model B trained.


# Cell 13 — Threshold selection on validation

In [16]:
def select_top_k_operating_point(model, X_val, y_val, min_recall=0.6):
    """
    Select top-K accounts on validation by descending predicted risk.

    K is selected using validation data only.
    No test information is used.
    """

    y_proba = model.predict_proba(X_val)[:, 1]
    y_true = y_val.to_numpy()

    sorted_indices = np.argsort(-y_proba)
    sorted_proba = y_proba[sorted_indices]

    records = []

    for k in range(1, len(y_true) + 1):

        y_pred = np.zeros(len(y_true), dtype=int)
        y_pred[sorted_indices[:k]] = 1

        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        cost = (
            fp * COST_FALSE_POSITIVE
            + fn * COST_FALSE_NEGATIVE
        )

        records.append({
            "k": k,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "cost": float(cost),
            "prob_cutoff": float(sorted_proba[k - 1]),
        })

    eligible = [
        r for r in records
        if r["recall"] >= min_recall
    ]

    if eligible:
        # Among recall-feasible operating points,
        # choose minimum expected cost.
        best = min(
            eligible,
            key=lambda r: (r["cost"], r["k"])
        )
    else:
        # Diagnostic fallback only.
        best = max(
            records,
            key=lambda r: (r["f1"], -r["cost"])
        )

    return best, records, y_proba

In [17]:
X_val_A, y_val_A = inputs["val_day4"]
X_val_B, y_val_B = inputs["val_graph"]

best_val_A, topk_records_A, _ = select_top_k_operating_point(model_A, X_val_A, y_val_A)
best_val_B, topk_records_B, _ = select_top_k_operating_point(model_B, X_val_B, y_val_B)

K_A = best_val_A["k"]
K_B = best_val_B["k"]

print("Model A validation-selected top-K:", K_A)
print(f"  val recall: {best_val_A['recall']:.4f}, val precision: {best_val_A['precision']:.4f}, val cost: ₹{best_val_A['cost']:,.0f}")

print("Model B validation-selected top-K:", K_B)
print(f"  val recall: {best_val_B['recall']:.4f}, val precision: {best_val_B['precision']:.4f}, val cost: ₹{best_val_B['cost']:,.0f}")

Model A validation-selected top-K: 195
  val recall: 1.0000, val precision: 0.0308, val cost: ₹378,000
Model B validation-selected top-K: 7
  val recall: 1.0000, val precision: 0.8571, val cost: ₹2,000


# Cell 14 — Final test evaluation

In [18]:
def evaluate_model_topk(model, X_test, y_test, k):
    """
    Evaluate model on test data by flagging exactly top-K accounts.

    K was selected previously using validation data only.
    """

    y_proba = model.predict_proba(X_test)[:, 1]
    y_true = y_test.to_numpy()

    sorted_indices = np.argsort(-y_proba)

    y_pred = np.zeros(len(y_true), dtype=int)
    y_pred[sorted_indices[:k]] = 1

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        y_proba
    )

    pr_auc = average_precision_score(
        y_true,
        y_proba
    )

    cost = (
        fp * COST_FALSE_POSITIVE
        + fn * COST_FALSE_NEGATIVE
    )

    return {
        "k": int(k),
        "y_proba": y_proba,
        "y_pred": y_pred,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "cost": float(cost),
    }

In [19]:
X_test_A, y_test_A = inputs["test_day4"]
X_test_B, y_test_B = inputs["test_graph"]

metrics_A = evaluate_model_topk(model_A, X_test_A, y_test_A, K_A)
metrics_B = evaluate_model_topk(model_B, X_test_B, y_test_B, K_B)

print("Model A test results (top-K):")
print(f"  K={metrics_A['k']}, Precision: {metrics_A['precision']:.4f}, Recall: {metrics_A['recall']:.4f}, F1: {metrics_A['f1']:.4f}")
print(f"  PR-AUC: {metrics_A['pr_auc']:.4f}, ROC-AUC: {metrics_A['roc_auc']:.4f}, Cost: ₹{metrics_A['cost']:,.0f}")

print("\nModel B test results (top-K):")
print(f"  K={metrics_B['k']}, Precision: {metrics_B['precision']:.4f}, Recall: {metrics_B['recall']:.4f}, F1: {metrics_B['f1']:.4f}")
print(f"  PR-AUC: {metrics_B['pr_auc']:.4f}, ROC-AUC: {metrics_B['roc_auc']:.4f}, Cost: ₹{metrics_B['cost']:,.0f}")

Model A test results (top-K):
  K=195, Precision: 0.0513, Recall: 1.0000, F1: 0.0976
  PR-AUC: 0.8244, ROC-AUC: 0.9965, Cost: ₹370,000

Model B test results (top-K):
  K=7, Precision: 0.5714, Recall: 0.4000, F1: 0.4706
  PR-AUC: 0.6632, ROC-AUC: 0.9902, Cost: ₹96,000


# Cell 15 — Rule baseline on same test set

In [20]:
def apply_rule_baseline(X_test):
    r1 = (X_test["return_rate"] > 0.5) & (X_test["total_orders"] >= 2)
    r2 = (X_test["shared_device_count"] >= 1) & (X_test["return_rate"] > 0.3)
    r3 = (
        X_test["account_creation_burst_score"] >= 5
    ) & (X_test["coupon_usage_rate"] > 0.5)
    r4 = (
        X_test["community_size"] >= 4
    ) & (X_test["community_return_rate"] > 0.4)
    r5 = X_test["dispute_rate"] > 0.3

    return (r1 | r2 | r3 | r4 | r5).astype(int)

In [21]:
X_test_B_baseline, y_test_B_baseline = inputs["test_graph"]

y_pred_baseline = apply_rule_baseline(X_test_B_baseline)

tn, fp, fn, tp = confusion_matrix(
    y_test_B_baseline, y_pred_baseline, labels=[0, 1]
).ravel()

baseline_metrics = {
    "precision": precision_score(y_test_B_baseline, y_pred_baseline, zero_division=0),
    "recall": recall_score(y_test_B_baseline, y_pred_baseline, zero_division=0),
    "f1": f1_score(y_test_B_baseline, y_pred_baseline, zero_division=0),
    "tp": int(tp),
    "fp": int(fp),
    "tn": int(tn),
    "fn": int(fn),
    "cost": fp * COST_FALSE_POSITIVE + fn * COST_FALSE_NEGATIVE,
}

print("Rule baseline test results:")
print(f"  Precision: {baseline_metrics['precision']:.4f}")
print(f"  Recall:    {baseline_metrics['recall']:.4f}")
print(f"  F1:        {baseline_metrics['f1']:.4f}")
print(f"  Cost:      ₹{baseline_metrics['cost']:,.0f}")

Rule baseline test results:
  Precision: 0.4375
  Recall:    0.7000
  F1:        0.5385
  Cost:      ₹63,000


# Cell 16 — Model comparison summary

In [22]:
summary = {
    "model_A": {
        "operating_k": metrics_A["k"],
        "precision": metrics_A["precision"],
        "recall": metrics_A["recall"],
        "f1": metrics_A["f1"],
        "pr_auc": metrics_A["pr_auc"],
        "roc_auc": metrics_A["roc_auc"],
        "tp": metrics_A["tp"],
        "fp": metrics_A["fp"],
        "tn": metrics_A["tn"],
        "fn": metrics_A["fn"],
        "cost": metrics_A["cost"],
    },

    "model_B": {
        "operating_k": metrics_B["k"],
        "precision": metrics_B["precision"],
        "recall": metrics_B["recall"],
        "f1": metrics_B["f1"],
        "pr_auc": metrics_B["pr_auc"],
        "roc_auc": metrics_B["roc_auc"],
        "tp": metrics_B["tp"],
        "fp": metrics_B["fp"],
        "tn": metrics_B["tn"],
        "fn": metrics_B["fn"],
        "cost": metrics_B["cost"],
    },

    "baseline": baseline_metrics,

    "validation": {
        "operating_k_A": K_A,
        "operating_k_B": K_B,
        "validation_recall_A": best_val_A["recall"],
        "validation_recall_B": best_val_B["recall"],
        "validation_precision_A": best_val_A["precision"],
        "validation_precision_B": best_val_B["precision"],
        "validation_cost_A": best_val_A["cost"],
        "validation_cost_B": best_val_B["cost"],
    },
}

print("===================================")
print("MODEL COMPARISON")
print("===================================")

print(
    f"Model A PR-AUC: {metrics_A['pr_auc']:.4f}"
)

print(
    f"Model B PR-AUC: {metrics_B['pr_auc']:.4f}"
)

print(
    f"Model A F1: {metrics_A['f1']:.4f}"
)

print(
    f"Model B F1: {metrics_B['f1']:.4f}"
)

print(
    f"Model A cost: ₹{metrics_A['cost']:,.0f}"
)

print(
    f"Model B cost: ₹{metrics_B['cost']:,.0f}"
)

print(
    f"Baseline cost: ₹{baseline_metrics['cost']:,.0f}"
)

print(
    f"\nOperating K — Model A: {K_A}"
)

print(
    f"Operating K — Model B: {K_B}"
)

MODEL COMPARISON
Model A PR-AUC: 0.8244
Model B PR-AUC: 0.6632
Model A F1: 0.0976
Model B F1: 0.4706
Model A cost: ₹370,000
Model B cost: ₹96,000
Baseline cost: ₹63,000

Operating K — Model A: 195
Operating K — Model B: 7


# Cell 17 — Feature importance

In [23]:
importance_B = model_B.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_cols_B,
    "importance": importance_B,
}).sort_values("importance", ascending=False)

print("Top 15 important features:")
print(importance_df.head(15).to_string(index=False))

Top 15 important features:
                     feature  importance
            account_age_days         471
      clustering_coefficient         335
      shared_ip_prefix_count         292
account_creation_burst_score         201
   community_avg_order_value         199
      shared_edge_weight_sum         198
       community_return_rate         182
                total_amount         129
                 refund_rate         116
         shared_device_count         110
             avg_order_value         103
      eigenvector_centrality         101
        accounts_per_address          96
               total_returns          89
           degree_centrality          89


# Cell 18 — Save all artifacts

In [24]:
with open(MODEL_PATH_A, "wb") as f:
    pickle.dump(model_A, f)

with open(MODEL_PATH_B, "wb") as f:
    pickle.dump(model_B, f)

# Predictions on test
test_predictions = pd.DataFrame({
    "account_id": split_data["test"]["graph"]["account_id"],
    "true_label": y_test_B.values,
    "proba_A": metrics_A["y_proba"],
    "proba_B": metrics_B["y_proba"],
    "pred_A": metrics_A["y_pred"],
    "pred_B": metrics_B["y_pred"],
    "pred_baseline": y_pred_baseline,
})
test_predictions.to_csv(PREDICTIONS_TEST_PATH, index=False)

# Metrics JSON
with open(MODEL_METRICS_PATH, "w") as f:
    json.dump(summary, f, indent=2)

# Feature importance
importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

print("Artifacts saved.")

Artifacts saved.


# Cell 19 — Leakage report

In [25]:
report = f"""RingWatch — Model Leakage Report
====================================

Prediction cutoff:
{T}

Train/Validation/Test split:
  Train:        {RING_TRAIN} + {len(normal_train)} non-ring accounts
  Validation:   {RING_VAL} + {len(normal_val)} non-ring accounts
  Test:         {RING_TEST} + {len(normal_test)} non-ring accounts

Train positives:        {int(y_train_A.sum())}
Validation positives:   {int(y_val_A.sum())}
Test positives:         {int(y_test_B.sum())}

Ring-aware split: YES
Top-K operating point selected on validation only: YES
Ground truth used only after feature generation: YES
No future events used: YES

Model A: Day 4 behavioral + identity features only.
Model B: Day 5 behavioral + identity + graph features.

Probability threshold calibration: NOT USED.
Rank-based top-K operating point: YES.

LEAKAGE CHECK: PASSED
"""

with open(MODEL_LEAKAGE_REPORT_PATH, "w") as f:
    f.write(report)

print(report)

RingWatch — Model Leakage Report

Prediction cutoff:
2026-02-20 00:00:00

Train/Validation/Test split:
  Train:        ['R001', 'R002'] + 478 non-ring accounts
  Validation:   ['R003'] + 191 non-ring accounts
  Test:         ['R004'] + 287 non-ring accounts

Train positives:        28
Validation positives:   6
Test positives:         10

Ring-aware split: YES
Top-K operating point selected on validation only: YES
Ground truth used only after feature generation: YES
No future events used: YES

Model A: Day 4 behavioral + identity features only.
Model B: Day 5 behavioral + identity + graph features.

Probability threshold calibration: NOT USED.
Rank-based top-K operating point: YES.

LEAKAGE CHECK: PASSED



# Cell 20 — Final stop condition

In [26]:
passed = True

if not (0 < metrics_B["recall"] < 1):
    print("Stop: Model B recall still trivial.")
    passed = False

if not (0 < metrics_B["precision"] < 1):
    print("Stop: Model B precision still trivial.")
    passed = False

if metrics_B["pr_auc"] <= 0.5:
    print("Warning: Model B PR-AUC is not above random.")
    passed = False

if not np.isfinite([metrics_B["cost"]]).all():
    print("Stop: Cost is not finite.")
    passed = False

if passed:
    print("\nDAY 6–7 COMPLETED")
    print("Model B has non-trivial predictive performance.")

    print("\nGraph ablation result:")
    print(f"  Model A PR-AUC = {metrics_A['pr_auc']:.4f}")
    print(f"  Model B PR-AUC = {metrics_B['pr_auc']:.4f}")
    print("  Graph features did NOT improve PR-AUC over Model A.")
    print("  This is recorded as a negative ablation result.")

    print("\nOperational comparison:")
    print(f"  Model B precision/F1/cost improved over Model A.")
    print(f"  Model B does not beat the Day-5 rule baseline on cost or F1.")
    print("\nProceed to Day 8 only for explainability/investigation analysis.")
else:
    print("\nDAY 6–7 DID NOT PASS")
    print("Do not proceed to Day 8 until fixed.")


DAY 6–7 COMPLETED
Model B has non-trivial predictive performance.

Graph ablation result:
  Model A PR-AUC = 0.8244
  Model B PR-AUC = 0.6632
  Graph features did NOT improve PR-AUC over Model A.
  This is recorded as a negative ablation result.

Operational comparison:
  Model B precision/F1/cost improved over Model A.
  Model B does not beat the Day-5 rule baseline on cost or F1.

Proceed to Day 8 only for explainability/investigation analysis.
